In [10]:
# Cargo las variables de entorno desde .env para que la API key de Open AI esté disponible
from dotenv import load_dotenv
load_dotenv()

True

In [11]:
# Creo el cliente de OpenAI que voy a usar para hacer llamadas a la API
from openai import OpenAI

openai_client = OpenAI()

In [12]:
# Defino la funcion llm() para hacer llamadas al modelo sin repetir codigo cada vez
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-4.1-mini",
        input=prompt
    )
    return response.output_text

In [13]:
llm("¿Cuál es la capital de Francia?")

'La capital de Francia es París.'

In [14]:
# Descargo las lesson pages del repo del curso desde un commit fijo
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [15]:
#Reviso que tiene cada uno de los archivos descargados
files[5]

RawRepositoryFile(filename='01-agentic-rag/lessons/06-building-prompt.md', content='# Building the Prompt\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=DV4e2n-dIv0&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nThe LLM doesn\'t see our documents unless we pass them in. So we need\nto build a prompt that includes the user\'s question and the search\nresults.\n\nWhen we build AI systems, we usually split the prompt into two parts:\n\n- Instructions (also called the system prompt): this tells the LLM how\n  to behave. It never changes, so it\'s the same for every request.\n- User prompt: this changes with every request. It carries the actual\n  question and the retrieved context.\n\nWe split them because the instructions are fixed and the user prompt is\nnot. Keeping them apart makes the fixed part easy to reuse and the\nchanging part easy to build fresh each time.\n\n## Instructions\n\nThe instructions tell the LLM its role and how to answer:\n\n```python\nINSTRUCTIONS = """

In [16]:
# Parseo cada archivo descargado y armo la lista de documentos
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

print(len(documents))

72


There are 72 lessons pages.

In [17]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [18]:
import minsearch

index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

In [19]:
results = index.search("How does the agentic loop keep calling the model until it stops?")

results[0]["filename"]

'01-agentic-rag/lessons/14-agentic-loop.md'

In [ ]:
import sys
sys.path.insert(0, "M1 lessons")
from rag_helper import RAGBase

class LessonRAG(RAGBase):

    # Sobreescribo search() porque mi i­ndice no tiene campos 'question', 'section' ni 'course'
    def search(self, query, num_results=5):
        return self.index.search(query, num_results=num_results)

    # Sobreescribo build_context() para construir el contexto con mis campos reales
    def build_context(self, search_results):
        lines = []
        for doc in search_results:
            lines.append(f"File: {doc['filename']}")
            lines.append(doc['content'])
            lines.append('')
        return '\n'.join(lines).strip()

    # Sobreescribo llm() para devolver la respuesta completa y poder leer el uso de tokens
    def llm(self, prompt):
        input_messages = [
            {'role': 'developer', 'content': self.instructions},
            {'role': 'user', 'content': prompt}
        ]
        response = self.llm_client.responses.create(
            model=self.model,
            input=input_messages
        )
        return response

    # Sobreescribo rag() para devolver tanto la respuesta como el uso de tokens
    def rag(self, query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        response = self.llm(prompt)
        return response.output_text, response.usage

In [21]:
# Instancio mi RAG con el i­ndice y el cliente de OpenAI que ya tengo en memoria
rag = LessonRAG(index=index, llm_client=openai_client)

# Corro la query de Q3 y capturo la respuesta y el uso de tokens
answer, usage = rag.rag("How does the agentic loop keep calling the model until it stops?")

print(answer)
print(usage.input_tokens)

The agentic loop keeps calling the model until it stops by running a `while True` loop that repeatedly sends the conversation history (including any function call results) back to the model and checks the model's response for function calls.

Here is how it works in detail:

1. Initialize a `has_function_calls` flag to `False` at the start of each iteration.
2. Send the current `messages` (which includes the developer instructions, user question, previous model messages, and any previous tool outputs) to the LLM.
3. Append the model's output to the conversation history (`messages`).
4. Check if the output contains any function call:
   - If yes, execute the corresponding function (e.g., `search`) with the arguments the model provided.
   - Append the function call output to `messages`.
   - Set `has_function_calls = True` to indicate the model wants to continue searching or calling tools.
5. If the model's output contains a normal message (answer or intermediate reasoning), keep track 

In [22]:
# Divido los documentos en chunks de 2000 caracteres con paso de 1000 (queda un overlap de 1000 caracteres entre chunks)
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

print(len(chunks))

295


In [23]:
# Indexo los chunks igual que los documentos: content como texto, filename como keyword
chunk_index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

chunk_index.fit(chunks)

In [24]:
# Apunto el RAG al i­ndice de chunks y corro la misma query que en Q3
rag_chunks = LessonRAG(index=chunk_index, llm_client=openai_client)

answer_chunks, usage_chunks = rag_chunks.rag("How does the agentic loop keep calling the model until it stops?")

print(usage_chunks.input_tokens)

2310


Consume 3x veces menos tokens que la consulta con los documentos completos